In [ ]:
RUN_DATE = "2026-10-27"
SOURCE_SYSTEM = "maven_global_electronics_retailer"

In [ ]:
#source contracts: source column -> Bronze column, in source file order
SOURCES = {
    "customers": {
        "source_file": "Customers.csv",
        "target_table": "bronze.customers_raw",
        "encoding": "windows-1252",
        "key_columns": ["customer_key"],
        "columns": {
            "CustomerKey": "customer_key",
            "Gender": "gender_raw",
            "Name": "customer_name",
            "City": "city",
            "State Code": "state_code_raw",
            "State": "state",
            "Zip Code": "zip_code_raw",
            "Country": "country",
            "Continent": "continent",
            "Birthday": "birthday_raw",
        },
    },
    "products": {
        "source_file": "Products.csv",
        "target_table": "bronze.products_raw",
        "encoding": "UTF-8",
        "key_columns": ["product_key"],
        "columns": {
            "ProductKey": "product_key",
            "Product Name": "product_name",
            "Brand": "brand",
            "Color": "color",
            "Unit Cost USD": "unit_cost_usd_raw",
            "Unit Price USD": "unit_price_usd_raw",
            "SubcategoryKey": "subcategory_key",
            "Subcategory": "subcategory",
            "CategoryKey": "category_key",
            "Category": "category",
        },
    },
    "stores": {
        "source_file": "Stores.csv",
        "target_table": "bronze.stores_raw",
        "encoding": "UTF-8",
        "key_columns": ["store_key"],
        "columns": {
            "StoreKey": "store_key",
            "Country": "country",
            "State": "state",
            "Square Meters": "square_meters_raw",
            "Open Date": "open_date_raw",
        },
    },
    "exchange_rates": {
        "source_file": "Exchange_Rates.csv",
        "target_table": "bronze.exchange_rates_raw",
        "encoding": "UTF-8",
        "key_columns": ["rate_date_raw", "currency_code_raw"],
        "columns": {
            "Date": "rate_date_raw",
            "Currency": "currency_code_raw",
            "Exchange": "exchange_rate_raw",
        },
    },
}

In [ ]:
from functools import reduce

from pyspark.sql import functions as F


def load_bronze(source_name, source):
    source_file = source["source_file"]
    source_path = (
        f"Files/bronze/{source_name}/landing/"
        f"ingestion_date={RUN_DATE}/{source_file}"
    )
    batch_id = f"{source_name}_initial_{RUN_DATE.replace('-', '')}_01"

    #read every value as a string; "" is the quote escape in these extracts
    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("encoding", source["encoding"])
        .option("quote", '"')
        .option("escape", '"')
        .csv(source_path)
    )

    expected_columns = list(source["columns"])
    assert raw_df.columns == expected_columns, (
        f"{source_file}: header {raw_df.columns} does not match {expected_columns}"
    )

    bronze_df = (
        raw_df.select(*[
            F.col(source_column).alias(target_column)
            for source_column, target_column in source["columns"].items()
        ])
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_source_path", F.lit(source_path))
        .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_ingestion_date", F.to_date(F.lit(RUN_DATE)))
        .withColumn("_ingested_at_utc", F.current_timestamp())
    )

    source_count = raw_df.count()

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(source["target_table"])
    )

    #validate
    target_df = spark.table(source["target_table"])
    key_columns = source["key_columns"]
    metadata_columns = [c for c in target_df.columns if c.startswith("_")]

    return {
        "source_name": source_name,
        "target_table": source["target_table"],
        "batch_id": batch_id,
        "source_rows": source_count,
        "target_rows": target_df.count(),
        "duplicate_key_groups": (
            target_df.groupBy(*key_columns).count()
            .filter(F.col("count") > 1)
            .count()
        ),
        "null_key_rows": target_df.filter(
            reduce(lambda a, b: a | b, [F.col(c).isNull() for c in key_columns])
        ).count(),
        "missing_metadata_rows": target_df.filter(
            reduce(lambda a, b: a | b, [F.col(c).isNull() for c in metadata_columns])
        ).count(),
    }

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

results = []
for source_name, source in SOURCES.items():
    result = load_bronze(source_name, source)
    results.append(result)
    print(f"Successfully wrote {result['target_table']}")

display(spark.createDataFrame(results))

In [ ]:
for result in results:
    name = result["source_name"]
    assert result["target_rows"] == result["source_rows"], f"{name}: row count mismatch"
    assert result["duplicate_key_groups"] == 0, f"{name}: duplicate keys"
    assert result["null_key_rows"] == 0, f"{name}: null keys"
    assert result["missing_metadata_rows"] == 0, f"{name}: missing metadata"

#accented customer names must survive the windows-1252 read
display(
    spark.table("bronze.customers_raw")
    .filter(F.col("customer_key").isin("400009", "400031", "400068"))
    .select("customer_key", "city", "state")
)

#escaped quotes must stay inside the product name
display(
    spark.table("bronze.products_raw")
    .filter(F.col("product_key") == "116")
    .select("product_key", "product_name", "brand", "unit_price_usd_raw")
)